# Week 3 - ML2 - Exercise 3: Hugging Face Embeddings + RAG

Week 2 built a first RAG pipeline. Week 3 opens that pipeline and swaps key components.
We reuse a familiar document setting so the main change is easy to see: embeddings now come from local Hugging Face models through `SentenceTransformer`.
For generation, use DeepInfra by default or Anthropic as an alternative instead of the Week 2 OpenAI LLM.

You will:
1. Load PDFs
2. Split text into chunks
3. Inspect Hugging Face embedding models
4. Build a FAISS index from local embeddings
5. Visualize retrieval with UMAP
6. Generate a grounded answer with a different LLM provider
7. Compare one embedding-model choice

Edit only cells marked `TODO`.

## Learning Goals

- Understand what an embedding model contributes to RAG
- Use Hugging Face / SentenceTransformer models for local embeddings
- Build cosine-similarity retrieval with FAISS
- Visualize retrieved chunks in embedding space
- Keep the embedding model and LLM provider as separate design choices

In [ ]:
import os
from collections import Counter
from pathlib import Path

import anthropic
import faiss
import matplotlib.pyplot as plt
import numpy as np
import umap.umap_ as umap
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

load_dotenv()

In [ ]:
DATA_DIR = Path("data")
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

if not PDF_FILES:
    DATA_DIR = Path("week3/data")
    PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

if not PDF_FILES:
    raise FileNotFoundError("No PDFs found in week3/data. Add PDF files first.")

PDF_FILES

## Step 1 - Read PDFs

We keep the document-loading step simple. Each PDF becomes one document with a source name and extracted text.

In [ ]:
def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    page_texts = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            page_texts.append(text)
    return "\n".join(page_texts)


documents = []
for pdf_path in PDF_FILES:
    documents.append(
        {
            "source": pdf_path.name,
            "text": read_pdf_text(pdf_path),
        }
    )

print("Documents:", len(documents))
print(documents[0]["text"][:500])

## Step 2 - Chunk Text (`TODO 1`)

Embeddings work best when each vector represents a focused piece of text.
Choose chunk settings that keep the chunks readable while preserving some overlap.

In [ ]:
# TODO 1
# Set CHUNK_SIZE and CHUNK_OVERLAP.

CHUNK_SIZE = 
CHUNK_OVERLAP = 

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunk_records = []
for doc in documents:
    chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunk_records.append(
            {
                "source": doc["source"],
                "chunk_id": f"{doc['source']}_chunk_{i}",
                "chunk_text": chunk,
            }
        )

chunk_records[:5]

In [ ]:
chunk_count_by_source = Counter(item["source"] for item in chunk_records)

print("Total chunks:", len(chunk_records))
print(chunk_count_by_source)
print("\nFirst chunk preview:\n")
print(chunk_records[0]["chunk_text"][:500])

## Step 3 - Inspect Hugging Face Embedding Models

A Hugging Face embedding model converts text into vectors.
Different models can have different embedding dimensions, tokenizers, language coverage, speed, and retrieval behavior.

The first run may download the models from Hugging Face.

In [ ]:
MODEL_OPTIONS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

sample_text = chunk_records[0]["chunk_text"][:500]
MODEL_OPTIONS

In [ ]:
def inspect_embedding_model(model_name: str, text: str):
    preview_model = SentenceTransformer(model_name)
    tokens = preview_model.tokenizer.tokenize(text)[:20]
    vector = preview_model.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    return {
        "model": model_name,
        "embedding_dimension": vector.shape[1],
        "first_tokens": tokens,
    }


for model_name in MODEL_OPTIONS:
    info = inspect_embedding_model(model_name, sample_text)
    print("=" * 80)
    print("Model:", info["model"])
    print("Embedding dimension:", info["embedding_dimension"])
    print("First tokens:", info["first_tokens"])

In [ ]:
# TODO 2
# Choose one Hugging Face embedding model from MODEL_OPTIONS for the main index.

EMBEDDING_MODEL_NAME = ""
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Using embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

In [ ]:
chunk_texts = [item["chunk_text"] for item in chunk_records]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print("Chunk embeddings shape:", chunk_embeddings.shape)
print("Vector norm of first chunk:", np.linalg.norm(chunk_embeddings[0]))

## Step 4 - Build a FAISS Retriever

Because the vectors are normalized, inner product in FAISS behaves like cosine similarity.
Higher scores mean more similar chunks.

In [ ]:
faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)

print("Number of vectors in FAISS:", faiss_index.ntotal)

In [ ]:
def embed_query(query: str):
    return embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")


def retrieve_chunks(query: str, top_k: int = 3):
    query_vector = embed_query(query)
    k = min(top_k, len(chunk_texts))
    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = chunk_records[int(idx)]
        results.append(
            {
                "index": int(idx),
                "source": item["source"],
                "chunk_id": item["chunk_id"],
                "similarity": float(score),
                "chunk_text": item["chunk_text"],
            }
        )
    return results

In [ ]:
# TODO 3
# Test retrieval with two specific questions.

TOP_K = 
query_1 = ""
query_2 = ""

for query in [query_1, query_2]:
    print("=" * 80)
    print("Query:", query)
    results = retrieve_chunks(query, top_k=TOP_K)
    for item in results:
        print(f"- {item['chunk_id']} | similarity={item['similarity']:.4f}")
        print(f"  {item['chunk_text'][:180]}...")

## Step 5 - Visualize Retrieval With UMAP

UMAP compresses the embedding vectors into 2D so we can inspect retrieval behavior visually.
The plot is only a rough diagnostic, not proof that retrieval is correct.

In [ ]:
n_neighbors = min(10, max(2, len(chunk_embeddings) - 1))
reducer = umap.UMAP(
    n_neighbors=n_neighbors,
    metric="cosine",
    random_state=0,
    transform_seed=0,
)

projected_chunks = reducer.fit_transform(chunk_embeddings)
print("Projected chunk coordinates:", projected_chunks.shape)

In [ ]:
def plot_retrieval(query: str, top_k: int = 3):
    results = retrieve_chunks(query, top_k=top_k)
    query_vector = embed_query(query)
    projected_query = reducer.transform(query_vector)
    result_indices = [item["index"] for item in results]

    plt.figure(figsize=(9, 6))
    plt.scatter(
        projected_chunks[:, 0],
        projected_chunks[:, 1],
        s=24,
        color="lightgray",
        label="All chunks",
    )
    plt.scatter(
        projected_chunks[result_indices, 0],
        projected_chunks[result_indices, 1],
        s=90,
        facecolors="none",
        edgecolors="tab:green",
        linewidths=2,
        label="Retrieved chunks",
    )
    plt.scatter(
        projected_query[:, 0],
        projected_query[:, 1],
        s=120,
        marker="X",
        color="tab:red",
        label="Query",
    )

    for item in results:
        x, y = projected_chunks[item["index"]]
        plt.annotate(item["chunk_id"].split(".pdf")[0], (x, y), fontsize=8)

    plt.title(query)
    plt.legend()
    plt.show()

    return results

In [ ]:
# TODO 4
# Plot retrieval for one useful query.

plot_query = ""
plot_results = plot_retrieval(plot_query, top_k=TOP_K)

## Step 6 - Add DeepInfra or Anthropic

The retrieval system now uses Hugging Face embeddings.
For generation, choose one of the providers practiced in Week 1: DeepInfra by default, or Anthropic as an alternative.

Set `DEEPINFRA_API_KEY` or `ANTHROPIC_API_KEY` in your `.env` file before running the answer step.
DeepInfra exposes an OpenAI-compatible API, so the client class is named `OpenAI`, but the request is sent to DeepInfra.

In [ ]:
# TODO 5
# Edit the prompt so answers stay grounded in the retrieved context.

PROMPT_TEMPLATE = """
You are a careful esports analyst.
Answer the question using ONLY the context below.
If the context is insufficient, say: "I don't know based on the provided context."
Keep the answer concise.

Context:
{context}

Question:
{question}
"""

In [ ]:
LLM_PROVIDER = "deepinfra"  # "deepinfra" or "anthropic"
DEEPINFRA_MODEL = os.getenv("DEEPINFRA_MODEL", "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8")
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")


def call_llm(prompt: str) -> str:
    provider = LLM_PROVIDER.lower().strip()

    if provider == "deepinfra":
        if not os.getenv("DEEPINFRA_API_KEY"):
            raise RuntimeError("Missing DEEPINFRA_API_KEY.")
        client = OpenAI(
            api_key=os.getenv("DEEPINFRA_API_KEY"),
            base_url="https://api.deepinfra.com/v1/openai",
        )
        response = client.chat.completions.create(
            model=DEEPINFRA_MODEL,
            messages=[
                {"role": "system", "content": "You answer only from the supplied context."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
        )
        return response.choices[0].message.content

    if provider == "anthropic":
        if not os.getenv("ANTHROPIC_API_KEY"):
            raise RuntimeError("Missing ANTHROPIC_API_KEY.")
        client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        message = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=700,
            temperature=0.2,
            system="You answer only from the supplied context.",
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(block.text for block in message.content if hasattr(block, "text"))

    raise ValueError("LLM_PROVIDER must be 'deepinfra' or 'anthropic'.")


def generate_answer(question: str, top_k: int = 3):
    if LLM_PROVIDER.lower().strip() not in {"deepinfra", "anthropic"}:
        raise RuntimeError(
            "Week 3 uses DeepInfra or Anthropic for the LLM, not the Week 2 OpenAI LLM."
        )

    retrieved = retrieve_chunks(question, top_k=top_k)
    context = "\n\n".join(item["chunk_text"] for item in retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    answer = call_llm(prompt)
    return answer, retrieved

In [ ]:
# TODO 6
# Ask one question and inspect both the answer and the retrieved chunks.

test_question = ""
answer, used_chunks = generate_answer(test_question, top_k=TOP_K)

print("Question:", test_question)
print("\nAnswer:\n", answer)
print("\nRetrieved chunks:")
for item in used_chunks:
    print(f"- {item['chunk_id']} | similarity={item['similarity']:.4f}")

## Step 7 - Compare One Embedding Model (`TODO 7`)

Embedding models are not interchangeable details.
They can change which chunks are retrieved, even when the LLM and prompt stay the same.

In [ ]:
def quick_retrieval_ids(model_name: str, query: str, top_k: int = 3):
    comparison_model = SentenceTransformer(model_name)
    comparison_embeddings = comparison_model.encode(
        chunk_texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    comparison_index = faiss.IndexFlatIP(comparison_embeddings.shape[1])
    comparison_index.add(comparison_embeddings)

    query_vector = comparison_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = comparison_index.search(query_vector, min(top_k, len(chunk_texts)))
    return [
        (chunk_records[int(idx)]["chunk_id"], float(score))
        for score, idx in zip(scores[0], indices[0])
        if idx != -1
    ]

In [ ]:
# TODO 7
# Choose the other embedding model and compare top chunk IDs for the same query.

ALTERNATIVE_EMBEDDING_MODEL = ""

print("Main model:", EMBEDDING_MODEL_NAME)
print([(item["chunk_id"], item["similarity"]) for item in retrieve_chunks(test_question, top_k=TOP_K)])

print("\nAlternative model:", ALTERNATIVE_EMBEDDING_MODEL)
print(quick_retrieval_ids(ALTERNATIVE_EMBEDDING_MODEL, test_question, top_k=TOP_K))

# Short note:
# Did the retrieved chunks change? Which model looked more useful for your question?

## Submission Checklist

- [ ] Chunking choice documented
- [ ] Hugging Face embedding model selected
- [ ] Retrieval tested with at least 2 meaningful queries
- [ ] UMAP retrieval plot created and interpreted
- [ ] DeepInfra or Anthropic answer generated from retrieved context
- [ ] One alternative embedding model compared
- [ ] Final reflection completed

## Final Reflection (max 8 lines)

1. What changed when you used Hugging Face embeddings instead of API embeddings?
2. Which retrieved chunk gave the strongest evidence for your answer?
3. Did the alternative embedding model retrieve different evidence?
4. What is one limitation of this RAG setup?